# ECG experiments for the journal version: baselines + repeated runs

Runs, in priority order (each result is saved to `experiments.csv` as soon as it finishes):

1. EfficientNetB0, seed 42: split-first (proposed) and augment-first
2. ResNet50, MobileNetV2, DenseNet121, seed 42: both protocols
3. EfficientNetB0 on the de-duplicated dataset (split-first, seed 42)
4. EfficientNetB0, seeds 1-4: both protocols

**How to run:** GPU T4 x2 on, then *Save Version → Save & Run All (Commit)*. It runs in the background (several hours) and you can close the browser.
When it finishes, open the version, go to Output, and download `experiments.csv`, `summary.json` and `fig_protocol_models.png`.

In [ ]:
import os, gc, time, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, roc_auc_score, cohen_kappa_score, recall_score
from sklearn.preprocessing import label_binarize

DATA_ROOT      = "/kaggle/input"
OUT            = "/kaggle/working"
IMG_SIZE       = 384
BATCH_SIZE     = 32
AUG_COPIES     = 3
CLASS_NAMES    = ["Abnormal_Heartbeat", "MI_history", "Myocardial_Infarction", "Normal"]
MI_CLASS       = CLASS_NAMES.index("Myocardial_Infarction")
TIME_BUDGET_H  = 11.0          # stop starting new runs after this many hours (Kaggle limit is 12 h)
RESULTS_CSV    = os.path.join(OUT, "experiments.csv")

if tf.config.list_physical_devices("GPU"):
    keras.mixed_precision.set_global_policy("mixed_float16")
print("TF", tf.__version__, "| GPUs:", len(tf.config.list_physical_devices("GPU")))

In [ ]:
# Load every image once (same loading as the main notebook)
def folder_to_class(name):
    n = name.lower()
    if "history" in n:                          return "MI_history"
    if "abnormal" in n:                         return "Abnormal_Heartbeat"
    if "myocardial" in n or "infarction" in n:  return "Myocardial_Infarction"
    if "normal" in n:                           return "Normal"
    return None

paths, labels = [], []
for root, _, files in os.walk(DATA_ROOT):
    cls = folder_to_class(os.path.basename(root))
    if cls:
        for f in sorted(files):
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                paths.append(os.path.join(root, f))
                labels.append(CLASS_NAMES.index(cls))

def load(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return tf.cast(tf.clip_by_value(tf.image.resize(img, (IMG_SIZE, IMG_SIZE), antialias=True), 0, 255), tf.uint8).numpy()

X_all  = np.stack([load(p) for p in paths])
y_all  = np.array(labels)
hashes = np.array([hashlib.md5(open(p, "rb").read()).hexdigest() for p in paths])
_, first_idx = np.unique(hashes, return_index=True)
DEDUP_IDX = np.sort(first_idx)
print("Images:", X_all.shape, "| per class:", np.bincount(y_all))
print("Unique images:", len(DEDUP_IDX), "| per class:", np.bincount(y_all[DEDUP_IDX]))

In [ ]:
# Data preparation for both protocols
def augment_copies(X):
    aug = keras.Sequential([
        layers.RandomTranslation(0.05, 0.05, fill_mode="constant", fill_value=255.0),
        layers.RandomZoom(0.1, fill_mode="constant", fill_value=255.0),
        layers.RandomRotation(0.02, fill_mode="constant", fill_value=255.0),
        layers.RandomContrast(0.2),
    ])
    copies = []
    for _ in range(AUG_COPIES):
        for i in range(0, len(X), 64):
            b = aug(tf.cast(X[i:i + 64], tf.float32), training=True)
            copies.append(tf.cast(tf.clip_by_value(b, 0, 255), tf.uint8).numpy())
    return np.concatenate([X] + copies)

def split_indices(y, seed):
    idx = np.arange(len(y))
    tr, tmp = train_test_split(idx, test_size=0.30, stratify=y, random_state=seed)
    va, te  = train_test_split(tmp, test_size=0.50, stratify=y[tmp], random_state=seed)
    return tr, va, te

def get_data(protocol, seed, dedup=False):
    X, y = (X_all[DEDUP_IDX], y_all[DEDUP_IDX]) if dedup else (X_all, y_all)
    if protocol == "split_first":            # proposed: split originals, augment training set only
        tr, va, te = split_indices(y, seed)
        Xtr = augment_copies(X[tr])
        ytr = np.concatenate([y[tr]] * (AUG_COPIES + 1))
        return Xtr, ytr, X[va], y[va], X[te], y[te]
    else:                                     # augment_first: augment everything, then split
        Xa = augment_copies(X)
        ya = np.concatenate([y] * (AUG_COPIES + 1))
        tr, va, te = split_indices(ya, seed)
        out = (Xa[tr], ya[tr], Xa[va], ya[va], Xa[te], ya[te])
        del Xa
        return out

In [ ]:
# Models: same head, augmentation and two-phase training for every backbone.
# Each backbone gets the input preprocessing its ImageNet weights expect.
def preprocessing(name):
    if name == "EfficientNetB0":            # has its own rescaling inside
        return []
    if name == "ResNet50":                  # caffe style: RGB->BGR, subtract mean
        return [layers.Lambda(lambda t: keras.ops.flip(t, axis=-1), name="rgb_to_bgr"),
                layers.Normalization(mean=[103.939, 116.779, 123.68], variance=[1.0, 1.0, 1.0])]
    if name == "MobileNetV2":               # scale to [-1, 1]
        return [layers.Rescaling(1.0 / 127.5, offset=-1.0)]
    if name == "DenseNet121":               # torch style: ImageNet mean/std
        return [layers.Normalization(mean=[123.675, 116.28, 103.53],
                                     variance=[58.395 ** 2, 57.12 ** 2, 57.375 ** 2])]
    raise ValueError(name)

BACKBONES = {
    "EfficientNetB0": keras.applications.EfficientNetB0,
    "ResNet50":       keras.applications.ResNet50,
    "MobileNetV2":    keras.applications.MobileNetV2,
    "DenseNet121":    keras.applications.DenseNet121,
}

def build_model(name):
    base = BACKBONES[name](include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(1.0)(inputs)
    x = layers.RandomTranslation(0.03, 0.03, fill_mode="constant", fill_value=255.0)(x)
    x = layers.RandomZoom(0.05, fill_mode="constant", fill_value=255.0)(x)
    x = layers.RandomContrast(0.15)(x)
    for layer in preprocessing(name):
        x = layer(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(len(CLASS_NAMES), activation="softmax", dtype="float32")(x)
    return keras.Model(inputs, outputs), base

def run(name, protocol, seed, dedup=False):
    keras.backend.clear_session(); gc.collect()
    keras.utils.set_random_seed(seed)
    Xtr, ytr, Xva, yva, Xte, yte = get_data(protocol, seed, dedup)
    model, base = build_model(name)
    Ytr = keras.utils.to_categorical(ytr, len(CLASS_NAMES))
    Yva = keras.utils.to_categorical(yva, len(CLASS_NAMES))
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.arange(len(CLASS_NAMES)), y=ytr)))

    def fit(lr, epochs, initial_epoch=0):
        model.compile(optimizer=keras.optimizers.Adam(lr), loss="categorical_crossentropy", metrics=["accuracy"])
        return model.fit(Xtr, Ytr, validation_data=(Xva, Yva), batch_size=BATCH_SIZE, verbose=2,
                         epochs=epochs, initial_epoch=initial_epoch, class_weight=cw,
                         callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
                                    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3)])
    t0 = time.time()
    h1 = fit(1e-3, 20)
    base.trainable = True
    for layer in base.layers[:-60]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    start = len(h1.history["loss"])
    h2 = fit(1e-4, start + 40, initial_epoch=start)

    probs  = model.predict(Xte, batch_size=BATCH_SIZE, verbose=0).astype("float64")
    y_pred = probs.argmax(1)
    row = {
        "model": name, "protocol": protocol, "dedup": dedup, "seed": seed,
        "n_train": len(ytr), "n_val": len(yva), "n_test": len(yte),
        "accuracy": float((y_pred == yte).mean()),
        "macro_f1": float(f1_score(yte, y_pred, average="macro")),
        "macro_auc": float(roc_auc_score(label_binarize(yte, classes=range(len(CLASS_NAMES))), probs, average="macro")),
        "kappa": float(cohen_kappa_score(yte, y_pred)),
        "mi_recall": float(recall_score(yte, y_pred, labels=[MI_CLASS], average="macro")),
        "best_val_loss": float(min(h1.history["val_loss"] + h2.history["val_loss"])),
        "epochs": start + len(h2.history["loss"]),
        "params": int(model.count_params()),
        "train_min": round((time.time() - t0) / 60, 1),
    }
    del Xtr, ytr, Xva, yva, Xte, yte, model, base
    gc.collect()
    return row

In [ ]:
# Quick self-test (about 1-2 minutes): build every model and run a forward pass.
# If this cell fails, stop and send me the error before spending GPU hours.
for name in BACKBONES:
    keras.backend.clear_session()
    m, _ = build_model(name)
    p = m.predict(X_all[:2].astype("float32"), verbose=0)
    assert p.shape == (2, len(CLASS_NAMES)) and np.allclose(p.sum(1), 1, atol=1e-3), name
    print(f"OK  {name:15s} params={m.count_params():,}")
    del m
keras.backend.clear_session(); gc.collect()

In [ ]:
# Experiment plan (priority order) and main loop. Results are appended after every run.
PLAN = [("EfficientNetB0", "split_first", 42, False), ("EfficientNetB0", "augment_first", 42, False)]
for m in ["ResNet50", "MobileNetV2", "DenseNet121"]:
    PLAN += [(m, "split_first", 42, False), (m, "augment_first", 42, False)]
PLAN += [("EfficientNetB0", "split_first", 42, True)]
for s in [1, 2, 3, 4]:
    PLAN += [("EfficientNetB0", "split_first", s, False), ("EfficientNetB0", "augment_first", s, False)]

key = lambda m, p, s, d: f"{m}|{p}|{s}|{d}"
done = set()
if os.path.exists(RESULTS_CSV):
    prev = pd.read_csv(RESULTS_CSV)
    done = {key(r.model, r.protocol, r.seed, r.dedup) for r in prev.itertuples()}
    print("Resuming, already done:", len(done))

T0 = time.time()
for i, (m, p, s, d) in enumerate(PLAN, 1):
    if key(m, p, s, d) in done:
        continue
    if (time.time() - T0) / 3600 > TIME_BUDGET_H:
        print("Time budget reached, stopping before:", m, p, s, d)
        break
    print(f"\n===== Run {i}/{len(PLAN)}: {m} | {p} | seed {s} | dedup {d} =====")
    try:
        row = run(m, p, s, d)
    except Exception as e:                      # one failed run must not stop the others
        print("RUN FAILED:", m, p, s, d, "->", repr(e))
        keras.backend.clear_session(); gc.collect()
        continue
    print({k: row[k] for k in ("model", "protocol", "seed", "dedup", "accuracy", "macro_f1", "train_min")})
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode="a", header=not os.path.exists(RESULTS_CSV), index=False)

print(f"\nAll done in {(time.time() - T0) / 3600:.2f} h")

In [ ]:
# Summary tables, figure and summary.json
df = pd.read_csv(RESULTS_CSV)
pd.set_option("display.width", 200)
print(df.round(4).to_string(index=False))

summary = {}
main = df[(df.model == "EfficientNetB0") & (~df.dedup)]
seeds_tab = main.groupby("protocol")[["accuracy", "macro_f1", "macro_auc", "kappa", "mi_recall"]].agg(["mean", "std", "count"])
print("\nEfficientNetB0 over seeds (mean, std, n):\n", seeds_tab.round(4))
summary["efficientnet_over_seeds"] = {
    p: {m: {"mean": float(g[m].mean()), "std": float(g[m].std(ddof=1)) if len(g) > 1 else None, "n": int(len(g))}
        for m in ["accuracy", "macro_f1", "macro_auc", "kappa", "mi_recall"]}
    for p, g in main.groupby("protocol")}

s42 = df[(df.seed == 42) & (~df.dedup)].pivot_table(index="model", columns="protocol", values="accuracy")
if {"split_first", "augment_first"} <= set(s42.columns):
    s42["gap"] = s42["augment_first"] - s42["split_first"]
print("\nSeed 42, accuracy by model and protocol:\n", (s42 * 100).round(2))
summary["seed42_by_model"] = df[(df.seed == 42) & (~df.dedup)].drop(columns=["dedup"]).to_dict(orient="records")
summary["dedup"] = df[df.dedup].to_dict(orient="records")

json.dump(summary, open(os.path.join(OUT, "summary.json"), "w"), indent=2)

# Figure: split-first vs augment-first per model (seed 42)
if {"split_first", "augment_first"} <= set(s42.columns):
    order = [m for m in ["EfficientNetB0", "ResNet50", "MobileNetV2", "DenseNet121"] if m in s42.index]
    xs, w = np.arange(len(order)), 0.38
    fig, ax = plt.subplots(figsize=(7.5, 4))
    b1 = ax.bar(xs - w/2, s42.loc[order, "split_first"] * 100, w, label="Split before augmentation (proposed)", color="#2F6DB5")
    b2 = ax.bar(xs + w/2, s42.loc[order, "augment_first"] * 100, w, label="Augment before split", color="#D9822B")
    ax.bar_label(b1, fmt="%.1f", fontsize=9); ax.bar_label(b2, fmt="%.1f", fontsize=9)
    ax.set_xticks(xs, order); ax.set_ylabel("Test accuracy (%)")
    ax.set_ylim(max(0, min(s42.loc[order, ["split_first", "augment_first"]].min()) * 100 - 10), 101)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, loc="lower right")
    plt.savefig(os.path.join(OUT, "fig_protocol_models.png"), dpi=300, bbox_inches="tight")
    plt.show()

print("\nDownload experiments.csv, summary.json and fig_protocol_models.png from the Output tab.")